# ODE Driver

This notebook loads `config.yaml`, defines a system, solves it using the helper library in `Helpers/`, and creates phase-space and Poincaré plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sage.all as sage
from sage.all import tanh

from Helpers.ode_driver import *

config = load_driver_config('config.yaml')

print('Config loaded successfully')
print('System settings:', config.get('system', {}))
print('Integration settings:', config.get('integration', {}))
print('Plot settings:', config.get('plot_3d', {}))
print('Poincaré settings:', config.get('poincare', {}))

In [ ]:
# Define your system here
# For a first-order system, return a list of N_VARS derivatives.

def system_func(t, state):
        P, E, M = state

        carrying_cap = 1.0

        pop_growth_rate = 0.3
        capture_rate = 0.4
        death_rate = 0.15

        eco_growth_rate = 0.3
        mil_budget = 0.2

        mil_pop = 1.0
        mob_pop = 0.333

        dP = carrying_cap * tanh(pop_growth_rate * E - capture_rate * M) - death_rate * P
        dE = carrying_cap * tanh(capture_rate * P) + mil_budget * M - eco_growth_rate * E
        dM = mil_pop * tanh(capture_rate * P + mil_budget * (M - E)) - mob_pop * tanh(P) * tanh(E)

        return [dP, dE, dM]

# If using a higher-order system, uncomment and define highest_derivative instead.
# def highest_derivative(t, state):
#     return ...

In [ ]:
# Build the ODE system and solve the initial-condition grid

ic_grid = build_initial_conditions(config)
print(f'Using {len(ic_grid)} initial conditions')

system = build_system(config, system_func=system_func)
viewer = solve_initial_conditions(config, system, ic_grid)

print(f'Solved {len(viewer.solutions)} trajectories')

In [ ]:
# Plot the 3D phase-space trajectories
plot_phase_space(viewer, config)

In [ ]:
# Compute and display the Poincaré section
poincare_result = plot_poincare(viewer, config)
if poincare_result is None:
    print('Poincaré section disabled, no intersections, or no plot generated.')

animation_result = animate_poincare(viewer, config)
if animation_result is None:
    print('Animation disabled or not created.')